In [1]:
import pandas as pd
import pyspark
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
from pyspark.context import SparkContext
from pyspark.sql import functions as F
from pyspark.sql import types
from pyspark.sql.functions import col

In [2]:
pyspark.__file__

'/home/daniel/anaconda3/lib/python3.13/site-packages/pyspark/__init__.py'

### Connecting to GCS from local Spark cluster

In [3]:
#configuration
gcs_credentials = '/home/daniel/.google/credentials/google_credentials.json'

conf = SparkConf() \
    .setMaster('local[*]') \
    .setAppName('test') \
    .set("spark.jars", "/home/daniel/notebooks/gcs_conn/gcs-connector-hadoop3-2.2.5.jar") \
    .set("spark.hadoop.google.cloud.auth.service.account.enable", "true") \
    .set("spark.hadoop.google.cloud.auth.service.account.json.keyfile", gcs_credentials)

In [4]:
#context
sc = SparkContext(conf=conf)

hadoop_conf = sc._jsc.hadoopConfiguration()

hadoop_conf.set("fs.AbstractFileSystem.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")
hadoop_conf.set("fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
hadoop_conf.set("fs.gs.auth.service.account.json.keyfile", gcs_credentials)
hadoop_conf.set("fs.gs.auth.service.account.enable", "true")

26/03/09 23:46:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [5]:
#Creating Spark session
spark = SparkSession.builder \
        .config(conf= sc.getConf()) \
        .getOrCreate()

In [6]:
df_green = \
    spark.read \
    .option('header', 'true') \
    .option('recursiveFileLookup', 'true') \
    .parquet('gs://bucket-spark-2026/taxi_data_pq/green/')


In [11]:
df_green.show(10)

+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|VendorID|lpep_pickup_datetime|lpep_dropoff_datetime|store_and_fwd_flag|RatecodeID|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|ehail_fee|improvement_surcharge|total_amount|payment_type|trip_type|congestion_surcharge|
+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|       2| 2019-01-27 18:07:53|  2019-01-27 18:20:54|                 N|         1|          82|         160|              1|         2.91|       11.5|  0.0|    0.